In [60]:
#각종 함수 선언 셀, process_dataframe, generate_pivot_statistics, expand_window
import pandas as pd
import numpy as np
import gc
from pandas.api.types import is_categorical_dtype
import re
from pathlib import Path
import sys
from pathlib import Path
ROOT = Path.cwd().parents[0]   # 상위폴더를 ROOT로 넣음.
sys.path.append(str(ROOT))
from stats.v_affix_attach import v_affix_attach, add_v_no_by_merge_and_fallback
from stats.en_no_fix import fix_en_number_with_merge, make_en_no_sub #EN_No 수정

# 결과에 덧붙이는 정보 처리 함수
def process_dataframe(result_df: pd.DataFrame) -> pd.DataFrame:
    # ---- 기본 방어: Series가 들어오면 DF로 ----
    if isinstance(result_df, pd.Series):
        result_df = result_df.reset_index(name=result_df.name or "ID")

    # ---- 인덱스가 의미있는 경우만 풀기 (MultiIndex/Named index) ----
    if not isinstance(result_df.index, pd.RangeIndex) or result_df.index.name is not None:
        result_df = result_df.reset_index()

    def add_ep_flags(df: pd.DataFrame, form_col: str, prefix: str) -> pd.DataFrame:
        if form_col in df.columns and f'{prefix}_T' not in df.columns:
            re_tt = re.compile(r'(?:었었|았었)')
            re_t  = re.compile(r'(?:었|았)')
            re_m  = re.compile(r'(?:겠|겄)')

            s = df[form_col].astype('string').str.replace(' + ', '', regex=False)
            df[form_col] = s
            tt = s.str.contains(re_tt, na=False)
            df[f'{prefix}_TT'] = tt
            df[f'{prefix}_T']  = (~tt) & s.str.contains(re_t, na=False)
            df[f'{prefix}_M']  = s.str.contains(re_m, na=False)
        return df

    # EN No sub정보 덧붙이기
    if ("EN_form" in result_df.columns) and ("EN_No_sub" not in result_df.columns):
        result_df = make_en_no_sub(result_df)

    # EP / f_EP / sentence / pre / post
    result_df = add_ep_flags(result_df, 'EP_form', 'EP')
    result_df = add_ep_flags(result_df, 'f_EP_form', 'f_EP')
    result_df = add_ep_flags(result_df, 'sentence_f_EP_form', 'sentence_f_EP')
    result_df = add_ep_flags(result_df, 'prev_sentence_f_EP_form', 'prev_sentence_f_EP')
    result_df = add_ep_flags(result_df, 'next_sentence_f_EP_form', 'next_sentence_f_EP')
            
    # V_No 생성
    if 'V_form_0' in result_df.columns and 'V_No' not in result_df.columns:
        V_No_file = r"..\..\label\000.V_No_(2026).csv"
        result_df = add_v_no_by_merge_and_fallback(result_df, V_No_file, encoding='utf-8-sig')

    return result_df

#개선된 그룹바이 함수
def pivot_statistics(
    df: pd.DataFrame,
    index_columns,
    filter_fn=None,
):
    """
    단일 DF에서 index_columns 조합별 빈도(size)를 계산해 Series로 반환.
    - dropna=False: 키 NA 포함
    - observed=True: category 조합 폭발 방지
    - sort=False: 속도
    """
    if filter_fn is not None:
        df = filter_fn(df)

    try:
        s = (
            df.groupby(index_columns, dropna=False, observed=True, sort=False)
              .size()
              .rename("ID")
        )
    except TypeError:
        # dropna 파라미터 호환 이슈 대비 (구버전/특정 상황)
        SENTINEL = "__<NA>__"
        sub = df[index_columns].copy()
        for c in index_columns:
            sub[c] = sub[c].where(sub[c].notna(), SENTINEL)
        s = sub.value_counts(sort=False).rename("ID")
        s.index = pd.MultiIndex.from_tuples(
            [tuple(np.nan if x == SENTINEL else x for x in tup) for tup in s.index],
            names=index_columns
        )
    return s

# Null값 등으로 인해서 제외되는 값이 있는지 확인하는 함수
def debug_drop_reason(df, index_columns):
    n_all = len(df)
    n_key_na = df[index_columns].isna().any(axis=1).sum()     # 키에 NA 있는 행 수
    n_id_na = df['ID'].isna().sum() if 'ID' in df.columns else 0

    # pivot_table식 count가 실제로 세는 개수(대략적인 감산식)
    approx_pivot_count = df.loc[~df[index_columns].isna().any(axis=1), 'ID'].notna().sum()

    # 권장 경로: groupby + size (키 NA 포함)
    gb_size = df.groupby(index_columns, dropna=False, observed=True).size().sum()

    return {
        "len(df)": int(n_all),
        "키에 NA 있는 행": int(n_key_na),
        "ID가 NA인 행": int(n_id_na),
        "pivot_table(count) 합": int(approx_pivot_count),
        "groupby(size, dropna=False) 합": int(gb_size),
    }

#개수 출력함수: main
def pivot_statistics_to_df(
    df: pd.DataFrame,
    index_columns,
    filter_fn=None,
    process_fn=None,
) -> pd.DataFrame:
    s = pivot_statistics(df, index_columns, filter_fn=filter_fn)
    out = s.reset_index()  # columns: index_columns + ['ID']

    # ✅ 후처리는 여기서 DF에 직접 적용 (결과 작아진 뒤)
    if process_fn is not None:
        out = process_fn(out)

    return out

from datetime import datetime
print(f"함수 생성 완료 - {datetime.now().strftime("%Y.%m.%d_%H:%M:%S")}")

함수 생성 완료 - 2026.03.22_23:39:26


In [5]:
from pathlib import Path
from datetime import datetime
START = datetime.now()

#대상 파일 읽어오기.
CSV_PATH = Path(r"..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver9.7.csv")
df_word = pd.read_csv(CSV_PATH)
print(f"*** 파일 읽기 완료: {CSV_PATH} ({len(df_word):,}행): {START.strftime("%Y.%m.%d_%H:%M:%S")}, during: {(datetime.now()-START)} ***")
df_word.columns

*** 파일 읽기 완료: ..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver9.7.csv (12,973,652행): 20260322_12-08-57, during: -1 day, 23:59:07.117566 ***


Index(['ID', 'file_id', 'docu_id', 'sen_id', 'word_id', 'rev_word_id',
       'form/label', 'N_form', 'N_label', 'V_No', 'V_form', 'V_label',
       'V_form_old', 'V_label_old', 'EP_form', 'EP_label', 'EN_form',
       'EN_label', 'J_form', 'J_label', 'EN_No', 'EN_No_sub', 'sent_end',
       'sent_end_V', 'sent_end_V_in_quote', 'quote_type', 'VX_len',
       'Next_VX_No', 'Next_VX_form', 'VX0_No', 'VX0_form', 'VX0_order',
       'V0_word_id', 'f_word_id'],
      dtype='object')

In [61]:
#문장 파일 읽기
SEN_CSV = r"..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver8.7_sen.csv"     
df_sen = pd.read_csv(SEN_CSV, low_memory=False)

In [6]:
df = df_word.copy()
df.columns

Index(['ID', 'file_id', 'docu_id', 'sen_id', 'word_id', 'rev_word_id',
       'form/label', 'N_form', 'N_label', 'V_No', 'V_form', 'V_label',
       'V_form_old', 'V_label_old', 'EP_form', 'EP_label', 'EN_form',
       'EN_label', 'J_form', 'J_label', 'EN_No', 'EN_No_sub', 'sent_end',
       'sent_end_V', 'sent_end_V_in_quote', 'quote_type', 'VX_len',
       'Next_VX_No', 'Next_VX_form', 'VX0_No', 'VX0_form', 'VX0_order',
       'V0_word_id', 'f_word_id'],
      dtype='object')

In [32]:
df[df["V_label"].isna()]['ID'].count()

np.int64(8389901)

In [29]:
df[df["V0_word_id"]==df["V0_word_id"]]["VX_len"].value_counts()

VX_len
0    12206964
1      685531
2       77506
3        5187
4         222
5           8
Name: count, dtype: int64

In [25]:
df[df["VX0_No"]==-1]['V_label'].value_counts()

V_label
VV    2548827
VA     879473
VP     372681
VX      16082
Name: count, dtype: int64

In [7]:
# ---------------------------------------------
# f_EN_no, f_EN_no_sub, f_EN_label 생성, V0_form, V0_label, V0_No 생성
# [주의]
# 이 df는 어절 단위 데이터프레임이다.
# 따라서 sentence_* / prev_sentence_* / next_sentence_* 컬럼은
# "현재 행"의 정보가 아니라 "현재 행이 속한 문장"의 문장 단위 정보를 반복 부착한 것이다.
#
# [문장끝 동사구]
# sent_end_V=True는 문장끝 동사구 전체에 부착될 수 있으므로,
# 한 문장에 여러 행이 해당될 수 있다.
# sentence_f_EP_form 등은 그중 대표행(현재는 첫 번째 행)의 값을 사용한 것이다.

# [대표행 선택]
# 문장 단위 분석을 df에서 수행할 때는 중복 제거 또는 대표행 선택이 필요하다.
# - 문장 대표행: df_sen 사용 권장
# - 문장끝 동사구의 맨 끝 어절(어미어절): (df["VX_len"] == 0) 또는 (df["VX0_No"] == -1)
# - 문장끝 동사구의 첫어절(동사어절):   (df['VX0_order']== -1) 또는 (df["V0_word_id"]==df["V0_word_id"]) 를 사용해야 한다. 

# [권장 필터]
# 필요에 따라 V_label.isna(), EN_label.isna() 등의 조건을 함께 사용해
# 분석 대상이 아닌 행이나 노이즈를 줄이는 것을 권장한다.
# ---------------------------------------------
START = datetime.now()
print(f"시작 - {START.strftime("%Y%m%d_%H-%M-%S")}")

# ---------------------------------------------
# 1. f_en_no, f_en_no_sub, f_en_label 생성
# ---------------------------------------------
if True: 
    df = df.drop(columns=["f_EN_form", "f_EN_label", "f_EN_No"], errors="ignore") # 기존에 있으면 제거 (중복 방지)

    # -1이면 자기 word_id2로 대체
    df["f_word_id"] = df["f_word_id"].where(
        df["f_word_id"] != -1,
        df["word_id"]
    )

    lookup = (
        df[
            [
                "sen_id",
                "word_id",
                "EN_form",
                "EN_No",
                "EN_No_sub",
                "EN_label",
                "EP_form"
            ]
        ]
        .rename(
            columns={
                "word_id": "f_word_id",
                "EN_form": "f_EN_form",
                "EN_No": "f_EN_No",
                "EN_No_sub": "f_EN_No_sub",
                "EN_label": "f_EN_label",
                "EP_form": "f_EP_form"
            }
        )
    )

    # ------------------------------------------
    # sen_id + f_word_id 기준 self merge
    # ------------------------------------------
    df = df.merge(
        lookup,
        on=["sen_id", "f_word_id"],
        how="left"
    )

print(f"f_EN 생성 완료 - {datetime.now()-START}")
# ---------------------------------------------
# 2. V0_form, V0_label, V0_No 생성
# ---------------------------------------------
START = datetime.now()
print(f"시작 - {START.strftime("%Y%m%d_%H-%M-%S")}")

if True:
    df = df.drop(columns=["V0_form", "V0_label", "V0_No"], errors="ignore") # 기존에 있으면 제거 (중복 방지)

    df["V0_word_id"] = df["V0_word_id"].where(
        df["V0_word_id"] != -1,
        df["word_id"]
    )

    lookup_V = (
        df[
            [
                "sen_id",
                "word_id",
                "V_form",
                "V_label",
                "V_No"
            ]
        ]
        .rename(
            columns={
                "word_id": "V0_word_id",
                "V_form": "V0_form",
                "V_label": "V0_label",
                "V_No": "V0_No"
            }
        )
    )

    df = df.merge(
        lookup_V,
        on=["sen_id", "V0_word_id"],
        how="left"
    )

print(f"V0 생성 완료 - {datetime.now()-START}")
df.columns

시작 - 20260322_12-10-41
f_EN 생성 완료 - 0:00:21.515240


In [58]:
df['speaker'].value_counts()

speaker
body    12747667
head      227746
Name: count, dtype: int64

In [35]:
# ==============================================
# 문장별 시제 정보 붙이기
# - sen_num, rev_sen_num은 df_sen에서 sen_id로 가져오기
# - 현재 문장, 이전 문장, 다음 문장의 문장끝 동사구 시제 정보 붙이기
# - 옵션: has_prev_sentence / has_next_sentence를 body만 기준으로 계산할지 선택
# ==============================================
import pandas as pd
from datetime import datetime

START = datetime.now()
print(f"시작 - {START.strftime('%Y%m%d_%H-%M-%S')}")

# -------------------------------------------------------------
# 옵션
# -------------------------------------------------------------
BODY_ONLY_FOR_SENTENCE_LINKS = True   # True면 body 문장만 기준으로 prev/next 계산

# -------------------------------------------------------------
# 문장 파일 읽기
# -------------------------------------------------------------
SEN_CSV = r"..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver8.7_sen.csv"
df_sen = pd.read_csv(SEN_CSV, low_memory=False)
print(f"***sentence 파일 읽기 완료: {SEN_CSV} ({len(df_sen):,}행): {datetime.now().strftime('%Y%m%d_%H-%M-%S')} ***")

# -----------------------------------
# 0. df_sen에서 sen_num / rev_sen_num 가져오기
# -----------------------------------
sen_cols = ["sen_id", "sen_num", "rev_sen_num"]

existing_sen_cols = [c for c in ["sen_num", "rev_sen_num"] if c in df.columns]
if existing_sen_cols:
    df = df.drop(columns=existing_sen_cols)

df = df.merge(
    df_sen[sen_cols].drop_duplicates(subset=["sen_id"]),
    on="sen_id",
    how="left"
)

# -----------------------------------
# 1. 문장별 기준표 만들기
#    -> body만 기준으로 prev/next를 계산할지 옵션 적용
# -----------------------------------
sentence_base = (
    df[["docu_id", "sen_id", "sen_num", "rev_sen_num", "speaker"]]
    .dropna(subset=["docu_id", "sen_id", "sen_num"])
    .drop_duplicates(subset=["docu_id", "sen_id"])
    .copy()
)

if BODY_ONLY_FOR_SENTENCE_LINKS:
    sentence_index = (
        sentence_base.loc[sentence_base["speaker"] == "body", ["docu_id", "sen_id", "sen_num", "rev_sen_num"]]
        .sort_values(["docu_id", "sen_num"])
        .copy()
    )
else:
    sentence_index = (
        sentence_base[["docu_id", "sen_id", "sen_num", "rev_sen_num"]]
        .sort_values(["docu_id", "sen_num"])
        .copy()
    )

# 이전 / 다음 문장 번호, 문장 ID
sentence_index["prev_sen_num"] = (
    sentence_index.groupby("docu_id")["sen_num"].shift(1)
)

sentence_index["next_sen_num"] = (
    sentence_index.groupby("docu_id")["sen_num"].shift(-1)
)

sentence_index["prev_sen_id"] = (
    sentence_index.groupby("docu_id")["sen_id"].shift(1)
)

sentence_index["next_sen_id"] = (
    sentence_index.groupby("docu_id")["sen_id"].shift(-1)
)

# 이전 / 다음 문장 존재 여부
sentence_index["has_prev_sentence"] = sentence_index["prev_sen_id"].notna()
sentence_index["has_next_sentence"] = sentence_index["next_sen_id"].notna()

# -----------------------------------
# 2. 각 문장의 문장끝 동사구 대표값 추출
#    sent_end_V=True인 것 중 첫 번째 f_EP_form만 사용
# -----------------------------------
sen_end_info = (
    df.loc[df["sent_end_V"] == True, ["docu_id", "sen_id", "sen_num", "f_EP_form", "sent_end_V_in_quote"]]
      .dropna(subset=["docu_id", "sen_id", "sen_num"])
      .sort_values(["docu_id", "sen_num"])
      .drop_duplicates(subset=["docu_id", "sen_id"], keep="first")
      .rename(columns={
          "f_EP_form": "sentence_f_EP_form",
          "sent_end_V_in_quote": "sentence_sent_end_V_in_quote"
      })
)

# -----------------------------------
# 3. 현재 문장의 문장끝 정보 붙이기
# -----------------------------------
sentence_index = sentence_index.merge(
    sen_end_info[["docu_id", "sen_id", "sentence_f_EP_form", "sentence_sent_end_V_in_quote"]],
    on=["docu_id", "sen_id"],
    how="left"
)

# -----------------------------------
# 4. 이전 문장의 정보 붙이기
# -----------------------------------
prev_info = sen_end_info[["docu_id", "sen_id", "sentence_f_EP_form", "sentence_sent_end_V_in_quote"]].rename(columns={
    "sen_id": "prev_sen_id",
    "sentence_f_EP_form": "prev_sentence_f_EP_form",
    "sentence_sent_end_V_in_quote": "prev_sentence_sent_end_V_in_quote"
})

sentence_index = sentence_index.merge(
    prev_info,
    on=["docu_id", "prev_sen_id"],
    how="left"
)

# -----------------------------------
# 5. 다음 문장의 정보 붙이기
# -----------------------------------
next_info = sen_end_info[["docu_id", "sen_id", "sentence_f_EP_form", "sentence_sent_end_V_in_quote"]].rename(columns={
    "sen_id": "next_sen_id",
    "sentence_f_EP_form": "next_sentence_f_EP_form",
    "sentence_sent_end_V_in_quote": "next_sentence_sent_end_V_in_quote"
})

sentence_index = sentence_index.merge(
    next_info,
    on=["docu_id", "next_sen_id"],
    how="left"
)

# -----------------------------------
# 6. 기존 컬럼 제거
# -----------------------------------
cols_to_drop = [
    "prev_sen_num",
    "next_sen_num",
    "prev_sen_id",
    "next_sen_id",
    "has_prev_sentence",
    "has_next_sentence",
    "sentence_f_EP_form",
    "prev_sentence_f_EP_form",
    "next_sentence_f_EP_form",
    "sentence_sent_end_V_in_quote",
    "prev_sentence_sent_end_V_in_quote",
    "next_sentence_sent_end_V_in_quote",
]

existing_cols_to_drop = [c for c in cols_to_drop if c in df.columns]
if existing_cols_to_drop:
    df = df.drop(columns=existing_cols_to_drop)

# -----------------------------------
# 7. 원래 df에 붙이기
# -----------------------------------
df = df.merge(
    sentence_index[
        [
            "docu_id",
            "sen_id",
            "sen_num",
            "rev_sen_num",
            "prev_sen_num",
            "next_sen_num",
            "prev_sen_id",
            "next_sen_id",
            "has_prev_sentence",
            "has_next_sentence",
            "sentence_f_EP_form",
            "prev_sentence_f_EP_form",
            "next_sentence_f_EP_form",
            "sentence_sent_end_V_in_quote",
            "prev_sentence_sent_end_V_in_quote",
            "next_sentence_sent_end_V_in_quote",
        ]
    ],
    on=["docu_id", "sen_id", "sen_num", "rev_sen_num"],
    how="left"
)

print(
    f"merge 완료 - {START.strftime('%Y%m%d_%H-%M-%S')}, "
    f"during: {datetime.now() - START}, "
    f"BODY_ONLY_FOR_SENTENCE_LINKS={BODY_ONLY_FOR_SENTENCE_LINKS}"
)

df.columns

시작 - 20260322_16-48-01
***sentence 파일 읽기 완료: ..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver8.7_sen.csv (1,048,127행): 20260322_16-48-08 ***
merge 완료 - 20260322_16-48-01, during: 0:01:56.646458, BODY_ONLY_FOR_SENTENCE_LINKS=True


Index(['ID', 'file_id', 'docu_id', 'sen_id', 'word_id', 'rev_word_id',
       'form/label', 'N_form', 'N_label', 'V_No', 'V_form', 'V_label',
       'V_form_old', 'V_label_old', 'EP_form', 'EP_label', 'EN_form',
       'EN_label', 'J_form', 'J_label', 'EN_No', 'EN_No_sub', 'sent_end',
       'sent_end_V', 'sent_end_V_in_quote', 'quote_type', 'VX_len',
       'Next_VX_No', 'Next_VX_form', 'VX0_No', 'VX0_form', 'VX0_order',
       'V0_word_id', 'f_word_id', 'f_EN_form', 'f_EN_No', 'f_EN_No_sub',
       'f_EN_label', 'f_EP_form', 'V0_form', 'V0_label', 'V0_No', 'speaker',
       'category', '매체', '내용', '파일제목', '저자', '출판사', '출판연도', 'head',
       'sen_count', 'sen_count_has_E', 'sen_count_not_quote',
       'sen_count_has_E_not_quote', 'base_count_not_quote', 'dominant_EN_No',
       'dominant_EN_label', 'dominant_count', 'dominant_ratio', 'sen_num',
       'rev_sen_num', 'prev_sen_num', 'next_sen_num', 'prev_sen_id',
       'next_sen_id', 'has_prev_sentence', 'has_next_sentence',
       '

In [12]:
df_sen.columns

Index(['file_id', 'docu_id', 'sen_id', 'file_sen_num', 'sen_num',
       'rev_sen_num', 'file_name', 'docu_no', 'sentence_form', 'speaker',
       'sen_len', 'quote_type', 'sent_has_E', 'last_EN_No', 'last_EN_label',
       'last_EN_in_quote', 'sentence_f_EP_form', 'quote_count',
       'quote_positions', 'quote_group_id', 'quote_span_len', 'long_quote',
       'quote_start_sen_id', 'quote_end_sen_id', 'span_wrap_ok', 'span_type',
       'passage_type', 'review_flag', 'review_note', 'quote_type_fix',
       'modified_at', 'quote_end_offset'],
      dtype='object')

In [53]:

df.head(1000)

,ID,file_id,docu_id,sen_id,word_id,rev_word_id,form/label,N_form,N_label,V_No,...,prev_sen_id,next_sen_id,has_prev_sentence,has_next_sentence,sentence_f_EP_form,prev_sentence_f_EP_form,next_sentence_f_EP_form,sentence_sent_end_V_in_quote,prev_sentence_sent_end_V_in_quote,next_sentence_sent_end_V_in_quote
0,AA0001.00005.0001,AA0001,AA0001.000,AA0001.00005,1,0,1993/SN + //SP + 06/SN + //SP + 08/SN,NaN,NaN,-1,...,NaN,AA0001.00006,False,True,NaN,NaN,NaN,NaN,NaN,NaN
1,AA0001.00006.0001,AA0001,AA0001.000,AA0001.00006,1,0,19/SN,NaN,NaN,-1,...,AA0001.00005,NaN,True,False,NaN,NaN,NaN,NaN,NaN,NaN
2,AA0001.00007.0001,AA0001,AA0001.001,AA0001.00007,1,-9,엠마누엘/NNP,NaN,NaN,-1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AA0001.00007.0002,AA0001,AA0001.001,AA0001.00007,2,-8,웅가/NNP + 로/JKB,웅가,NNP,-1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AA0001.00007.0003,AA0001,AA0001.001,AA0001.00007,3,-7,//SP,NaN,NaN,-1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,AA0001.00077.0014,AA0001,AA0001.005,AA0001.00077,14,-8,명/NNB + 으로/JKB,명,NNB,-1,...,AA0001.00076,AA0001.00078,True,True,았,NaN,NaN,False,False,NaN
996,AA0001.00077.0015,AA0001,AA0001.005,AA0001.00077,15,-7,",/SP",NaN,NaN,-1,...,AA0001.00076,AA0001.00078,True,True,았,NaN,NaN,False,False,NaN
997,AA0001.00077.0016,AA0001,AA0001.005,AA0001.00077,16,-6,파리/NNP + ∼/SO + 다카르/NNP,NaN,NaN,-1,...,AA0001.00076,AA0001.00078,True,True,았,NaN,NaN,False,False,NaN
998,AA0001.00077.0017,AA0001,AA0001.005,AA0001.00077,17,-5,"랠리/NNG + ,/SP",NaN,NaN,-1,...,AA0001.00076,AA0001.00078,True,True,았,NaN,NaN,False,False,NaN


In [14]:
stamp = datetime.now().strftime("%Y%m%d_%H-%M-%S")


#document 파일 읽기
DOCU_CSV = r"..\csv\original_csv\세종문어_document_정보.csv"
df_docu = pd.read_csv(DOCU_CSV, low_memory=False)

#file 파일 읽기
FILE_CSV = r"..\csv\original_csv\세종문어_file_정보.csv"
df_file = pd.read_csv(FILE_CSV, low_memory=False)

print(f"파일 읽기 완료 - {stamp}")
print(f"df_file: {df_file.columns.tolist()}")
print(f"df_docu: {df_docu.columns.tolist()}")
print(f"df_file: {len(df_file):,}행, df_docu: {len(df_docu):,}행, df_sen: {len(df_sen):,}행, df: {len(df):,}행")

파일 읽기 완료 - 20260322_12-17-17
df_file: ['file_id', '매체', '내용', '내용2', '파일제목', '저자', '출판사', '출판연도', '구어/문어', '분류기호', '분류기호2', '내용3', '분류기호4']
df_docu: ['docu_id', 'file_id', 'docu_num', 'category', '매체', '내용', '내용2', '파일제목', '저자', '출판사', '출판연도', 'head', '제목', '구어/문어', '분류기호', '분류기호2', '내용3', '분류기호4', 'sen_count', 'sen_count_has_E', 'sen_count_not_quote', 'sen_count_has_E_not_quote', 'base_count_not_quote', 'dominant_EN_No', 'dominant_EN_label', 'dominant_count', 'dominant_ratio']
df_file: 369행, df_docu: 33,155행, df_sen: 1,048,127행, df: 12,975,418행


In [57]:
print(f"총 행: {len(df):,}, N 수: {df['N_label'].count():,},  V 수: {df['V_label'].count():,}  EN 수: {df['EN_label'].count():,}")
print(f"V0수 : {df[(df['V_label'].notna()) & (df['VX0_order']==-1)]['ID'].count():,},       f_en수 : {df[(df['EN_label'].notna()) & (df['VX_len']==0)]['ID'].count():,}")

print("\n문장 끝")
print(f"문장의 마지막 V0수 : {df[(df['V_label'].notna()) & (df['VX0_order']==-1) & (df['sent_end_V']==True)]['ID'].count():,}, 문장의 마지막 f_en수 : {df[(df['EN_label'].notna()) & (df['VX_len']==0)& (df['sent_end_V']==True)]['ID'].count():,}")

print('\nbody만')
print(f"문장의 마지막 V0수 : {df[(df['V_label'].notna()) & (df['VX0_order']==-1) & (df['sent_end_V']==True)& (df['speaker']=='body')]['ID'].count():,}, 문장의 마지막 f_en수 : {df[(df['EN_label'].notna()) & (df['VX_len']==0)& (df['sent_end_V']==True)& (df['speaker']=='body')]['ID'].count():,}")

print('\nbody, has_prev_sentence')
print(f"문장의 마지막 V0수 : {df[(df['V_label'].notna()) & (df['VX0_order']==-1) & (df['sent_end_V']==True)& (df['speaker']=='body') &(df['has_prev_sentence'] == True)]['ID'].count():,}, 문장의 마지막 f_en수 : {df[(df['EN_label'].notna()) & (df['VX_len']==0)& (df['sent_end_V']==True)& (df['speaker']=='body')&(df['has_prev_sentence'] == True)]['ID'].count():,}")

print('\nbody, has_next_sentence')
print(f"문장의 마지막 V0수 : {df[(df['V_label'].notna()) & (df['VX0_order']==-1) & (df['sent_end_V']==True)& (df['speaker']=='body') &(df['has_next_sentence'] == True)]['ID'].count():,}, 문장의 마지막 f_en수 : {df[(df['EN_label'].notna()) & (df['VX_len']==0)& (df['sent_end_V']==True)& (df['speaker']=='body')&(df['has_next_sentence'] == True)]['ID'].count():,}")

총 행: 12,975,418, N 수: 5,051,557,  V 수: 4,585,517  EN 수: 4,599,152
V0수 : 3,817,063,       f_en수 : 3,832,248

문장 끝
문장의 마지막 V0수 : 979,425, 문장의 마지막 f_en수 : 982,446

body만
문장의 마지막 V0수 : 963,071, 문장의 마지막 f_en수 : 966,066

body, has_prev_sentence
문장의 마지막 V0수 : 931,121, 문장의 마지막 f_en수 : 934,005

body, has_next_sentence
문장의 마지막 V0수 : 939,878, 문장의 마지막 f_en수 : 942,810


In [59]:
# 문장 단위로 "has_prev_sentence","has_next_sentence" 확인.
sen_check = (
    df[["docu_id","sen_id","has_prev_sentence","has_next_sentence"]]
    .drop_duplicates()
)

print(
    sen_check["has_prev_sentence"].value_counts(),
    sen_check["has_next_sentence"].value_counts()
)

has_prev_sentence
True     973902
False     33152
Name: count, dtype: int64 has_next_sentence
True     973902
False     33152
Name: count, dtype: int64


In [36]:
# ---
# document 정보에서 필요한 컬럼 병합
# ---
def safe_merge(df, df_add, key, cols):
    cols_to_use = [key] + [c for c in cols if c not in df.columns]
    return df.merge(df_add[cols_to_use], on=key, how="left")

df = safe_merge(df, df_sen, "sen_id", ["speaker"])
df = safe_merge(
    df,
    df_docu,
    "docu_id",
    ['category', '매체', '내용', '파일제목', '저자', '출판사', '출판연도',
     'head', 'sen_count', 'sen_count_has_E', 'sen_count_not_quote',
       'sen_count_has_E_not_quote', 'base_count_not_quote', 'dominant_EN_No',
       'dominant_EN_label', 'dominant_count', 'dominant_ratio']
)

print(f"merge 완료 - {stamp}")

df.columns

merge 완료 - 20260322_12-28


Index(['ID', 'file_id', 'docu_id', 'sen_id', 'word_id', 'rev_word_id',
       'form/label', 'N_form', 'N_label', 'V_No', 'V_form', 'V_label',
       'V_form_old', 'V_label_old', 'EP_form', 'EP_label', 'EN_form',
       'EN_label', 'J_form', 'J_label', 'EN_No', 'EN_No_sub', 'sent_end',
       'sent_end_V', 'sent_end_V_in_quote', 'quote_type', 'VX_len',
       'Next_VX_No', 'Next_VX_form', 'VX0_No', 'VX0_form', 'VX0_order',
       'V0_word_id', 'f_word_id', 'f_EN_form', 'f_EN_No', 'f_EN_No_sub',
       'f_EN_label', 'f_EP_form', 'V0_form', 'V0_label', 'V0_No', 'speaker',
       'category', '매체', '내용', '파일제목', '저자', '출판사', '출판연도', 'head',
       'sen_count', 'sen_count_has_E', 'sen_count_not_quote',
       'sen_count_has_E_not_quote', 'base_count_not_quote', 'dominant_EN_No',
       'dominant_EN_label', 'dominant_count', 'dominant_ratio', 'sen_num',
       'rev_sen_num', 'prev_sen_num', 'next_sen_num', 'prev_sen_id',
       'next_sen_id', 'has_prev_sentence', 'has_next_sentence',
       '

In [37]:
OUT_DIR = r"..\csv\통계용"
PREFIX = "세종_문어_문장끝"
stamp = datetime.now().strftime("%Y%m%d_%H-%M-%S")

index_columns = ['category','매체', 'file_id', 'docu_id', 'speaker',
                 'sen_count', 'sen_count_has_E', 'sen_count_not_quote',
       'sen_count_has_E_not_quote', 'base_count_not_quote', 'dominant_EN_No',
       'dominant_EN_label', 'dominant_count', 'dominant_ratio',
       'V_No', 'V_form',
       'V_label', 'EP_form', 'EN_form', 'EN_label', 'EN_No', 'EN_No_sub', 'VX_len',
       'Next_VX_No', 'Next_VX_form', 'VX0_No', 'VX0_form', 'VX0_order',
       'V0_form', 'V0_No', 'V0_label',
       'f_EP_form', 'f_EN_form', 'f_EN_No',
       'f_EN_No_sub', 'f_EN_label',
       'has_prev_sentence', 'has_next_sentence',
       'sentence_f_EP_form', 'prev_sentence_f_EP_form',
       'next_sentence_f_EP_form', 
       'sentence_sent_end_V_in_quote', 'prev_sentence_sent_end_V_in_quote',
       'next_sentence_sent_end_V_in_quote']

FILTER_SEN_ENDS = lambda df: df[(df['sent_end_V'] == True)]
FILTER_SEN_ENDS_NOT_QUOTE = lambda df: df[(df['sent_end_V'] == True) & (df['sent_end_V_in_quote'] == False)]

#에러 확인
if True:
    print(debug_drop_reason(df, index_columns))

#실행
result_df = pivot_statistics_to_df(
    df =df,
    index_columns=index_columns,
    process_fn=process_dataframe,
    filter_fn=FILTER_SEN_ENDS
)
print(f"result_df length: {len(result_df):,}, now: {datetime.now().strftime("%Y.%m.%d_%H:%M:%S")}")
result_df.columns

{'len(df)': 12975418, '키에 NA 있는 행': 12975030, 'ID가 NA인 행': 0, 'pivot_table(count) 합': 388, 'groupby(size, dropna=False) 합': 12975418}
result_df length: 1,225,242, now: 2026.03.22_17:03:05


Index(['category', '매체', 'file_id', 'docu_id', 'speaker', 'sen_count',
       'sen_count_has_E', 'sen_count_not_quote', 'sen_count_has_E_not_quote',
       'base_count_not_quote', 'dominant_EN_No', 'dominant_EN_label',
       'dominant_count', 'dominant_ratio', 'V_No', 'V_form', 'V_label',
       'EP_form', 'EN_form', 'EN_label', 'EN_No', 'EN_No_sub', 'VX_len',
       'Next_VX_No', 'Next_VX_form', 'VX0_No', 'VX0_form', 'VX0_order',
       'V0_form', 'V0_No', 'V0_label', 'f_EP_form', 'f_EN_form', 'f_EN_No',
       'f_EN_No_sub', 'f_EN_label', 'has_prev_sentence', 'has_next_sentence',
       'sentence_f_EP_form', 'prev_sentence_f_EP_form',
       'next_sentence_f_EP_form', 'sentence_sent_end_V_in_quote',
       'prev_sentence_sent_end_V_in_quote',
       'next_sentence_sent_end_V_in_quote', 'ID', 'EP_TT', 'EP_T', 'EP_M',
       'f_EP_TT', 'f_EP_T', 'f_EP_M', 'sentence_f_EP_TT', 'sentence_f_EP_T',
       'sentence_f_EP_M', 'prev_sentence_f_EP_TT', 'prev_sentence_f_EP_T',
       'prev_sent

In [71]:
df = df_sen.copy()
df.columns

Index(['file_id', 'docu_id', 'sen_id', 'file_sen_num', 'sen_num',
       'rev_sen_num', 'file_name', 'docu_no', 'sentence_form', 'speaker',
       'sen_len', 'quote_type', 'sent_has_E', 'sentence_f_EN_No',
       'sentence_f_EN_label', 'sentence_f_EN_in_quote', 'sentence_f_EP_form',
       'quote_count', 'quote_positions', 'quote_group_id', 'quote_span_len',
       'long_quote', 'quote_start_sen_id', 'quote_end_sen_id', 'span_wrap_ok',
       'span_type', 'passage_type', 'review_flag', 'review_note',
       'quote_type_fix', 'modified_at', 'quote_end_offset'],
      dtype='object')

In [72]:
OUT_DIR = r"..\csv\통계용"
PREFIX = "document_구분1"
print(f"시작 - {datetime.now().strftime('%Y.%m.%d_%H:%M:%S')}")

index_columns =['file_id', 'docu_id', 'speaker',
       'quote_type', 'sent_has_E', 'sentence_f_EN_No',
       'sentence_f_EN_label', 'sentence_f_EN_in_quote', 'sentence_f_EP_form',]
df = df.rename(columns={
    'sen_id': "ID",
})
#에러 확인
if True:
    print(debug_drop_reason(df, index_columns))

#실행
result_df = pivot_statistics_to_df(
    df =df,
    index_columns=index_columns,
    process_fn=process_dataframe,
    #filter_fn=FILTER_SEN_ENDS
)
print(f"result_df length: {len(result_df):,}, now: {datetime.now().strftime('%Y.%m.%d_%H:%M:%S')}")
result_df.columns

시작 - 2026.03.22_23:56:24
{'len(df)': 1048127, '키에 NA 있는 행': 651404, 'ID가 NA인 행': 0, 'pivot_table(count) 합': 396723, 'groupby(size, dropna=False) 합': 1048127}
result_df length: 264,768, now: 2026.03.22_23:56:26


Index(['file_id', 'docu_id', 'speaker', 'quote_type', 'sent_has_E',
       'sentence_f_EN_No', 'sentence_f_EN_label', 'sentence_f_EN_in_quote',
       'sentence_f_EP_form', 'ID', 'sentence_f_EP_TT', 'sentence_f_EP_T',
       'sentence_f_EP_M'],
      dtype='object')

In [73]:
# 컬럼 순서 재배치
front_cols = []
last_cols = ['ID' ]
rest_cols = [col for col in result_df.columns if col not in front_cols + last_cols]
result_df = result_df[front_cols + rest_cols + last_cols]

# 컬럼명 변경
result_df = result_df.rename(columns={
    "ID": "count",
})
print(f"result_df length: {len(result_df):,}, now: {datetime.now().strftime("%Y.%m.%d_%H:%M:%S")}")
result_df.columns

result_df length: 264,768, now: 2026.03.22_23:56:40


Index(['file_id', 'docu_id', 'speaker', 'quote_type', 'sent_has_E',
       'sentence_f_EN_No', 'sentence_f_EN_label', 'sentence_f_EN_in_quote',
       'sentence_f_EP_form', 'sentence_f_EP_TT', 'sentence_f_EP_T',
       'sentence_f_EP_M', 'count'],
      dtype='object')

In [74]:
# 결과 DataFrame 저장
stamp = datetime.now().strftime("%Y%m%d_%H-%M")
save_folder = Path(OUT_DIR) 
output_file_name = save_folder / f"{PREFIX}_{stamp}.csv"
output_file_name.parent.mkdir(parents=True, exist_ok=True)  # 폴더 생성

result_df.to_csv(output_file_name, index=True, encoding='utf-8-sig')
print(f"*** 저장 완료: {output_file_name} ({len(result_df):,}행) ***")

*** 저장 완료: ..\csv\통계용\document_구분1_20260322_23-56.csv (264,768행) ***
